In [1]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Panama boundary
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

panama = countries.filter(
    ee.Filter.eq('country_na', 'Panama')
).geometry()

# Date range
START = '2024-01-01'
END = '2024-12-31'

# Dynamic World collection
dw_col = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterBounds(panama)
    .filterDate(START, END)
)

# Sentinel-2 collection
s2_col = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filterBounds(panama)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

print("DW images:", dw_col.size().getInfo())
print("S2 images:", s2_col.size().getInfo())

# Create mosaics/composites
# Dynamic World label mode composite
dw_mode = dw_col.select('label').mode()

# Sentinel-2 median composite
s2_median = s2_col.median()

# Visualization palettes
VIS_PALETTE = [
    '419bdf',  # water
    '397d49',  # trees
    '88b053',  # grass
    '7a87c6',  # flooded vegetation
    'e49635',  # crops
    'dfc35a',  # shrub/scrub
    'c4281b',  # built
    'a59b8f',  # bare
    'b39fe1',  # snow/ice
]

# Visualize Dynamic World
dw_vis = dw_mode.visualize(
    min=0,
    max=8,
    palette=VIS_PALETTE
)

# Map
m = geemap.Map()

m.centerObject(panama, 7)

# Sentinel-2 RGB
m.addLayer(
    s2_median,
    {
        'bands': ['B4', 'B3', 'B2'],
        'min': 0,
        'max': 3000
    },
    'Sentinel-2 Median'
)

# Dynamic World
m.addLayer(
    dw_vis,
    {},
    'Dynamic World Land Cover'
)

# Panama boundary
m.addLayer(
    panama,
    {'color': 'red'},
    'Panama Boundary'
)

m.addLayerControl()

m

DW images: 804
S2 images: 540


Map(center=[8.513881071215563, -80.10553238925849], controls=(WidgetControl(options=['position', 'transparent_…